# release16_official_replay_candidate_handoff_v1

Purpose: aggregate the latest release16 official replay candidates after the release-gate BC weighting repair and derive the next training/checkpoint objective. This is a model-selection handoff, not a promotion review.

Governance:
- Canonical baseline: `soxlbacktest_policy_reimplementation_v1`.
- Direct comparison: `release16_release_gate_bc_official_replay_v1`.
- Role: `diagnose`.
- Target windows: `wf_2020`, `wf_2022`, `wf_2023_2024`.
- Changed layer: `model_selection` / `checkpoint_objective` / `training_objective_handoff`.
- External reference: qlib-converted SOXLBacktest walk-forward baseline plus release16 checkpoint replay candidates.

Hypothesis: after fixing supervised release-gate fit, the remaining blocker is not BC loss weighting. A unified official replay candidate pool should show whether the next mechanism is drawdown control, underparticipation/trend capture, or a joint walk-forward objective before any PPO or retained actor run.

Primary gate in this notebook follows the user requirement: `relative_return >= +30pp` and `MaxDD <= 20%` on every walk-forward fold. The stricter `<=15%` historical gate is retained as stress context.


In [1]:

from __future__ import annotations

import json
import math
import sys
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from IPython.display import Markdown, display


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "SPEC.md").exists() and (candidate / "src/soxl_quant").exists():
            return candidate
    raise RuntimeError(f"could not locate repo root from {start}")

ROOT = find_repo_root(Path.cwd().resolve())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.soxl_quant.app.official_candidate_aggregation import (
    OfficialReplayCandidateConfig,
    normalise_official_replay_candidate_frame,
    score_official_replay_candidate_pool,
    score_official_replay_stress_scenarios,
)
from src.soxl_quant.app.training_objective_handoff import derive_training_objective_handoff, render_training_objective_handoff

EXPERIMENT = "release16_official_replay_candidate_handoff_v1"
CARD = ROOT / "runtime/model_selection/card" / EXPERIMENT
CARD.mkdir(parents=True, exist_ok=True)

PRIMARY_CONFIG = OfficialReplayCandidateConfig(target_relative_return_pp=30.0, target_max_dd_pct=20.0, max_dd_gap_weight=10.0)
STRICT_CONFIG = OfficialReplayCandidateConfig(target_relative_return_pp=30.0, target_max_dd_pct=15.0, max_dd_gap_weight=10.0)
WINDOWS = ["wf_2020", "wf_2022", "wf_2023_2024"]

CURRENT_REPLAY = ROOT / "runtime/model_selection/card/release16_release_gate_bc_official_replay_v1/bc_official_replay_metrics.csv"
CURRENT_SELECTION = ROOT / "runtime/model_selection/card/release16_release_gate_bc_official_replay_v1/selection.json"
WEIGHT_SCREEN = ROOT / "runtime/model_selection/card/release16_release_gate_bc_weight_screen_v1/summary.json"
PREV_CHECKPOINT = ROOT / "runtime/model_selection/card/release16_rulebase_reversal_drawdown_upside_checkpoint_metric_replay_v1/checkpoint_candidate_metrics.csv"
PREV_SELECTION = ROOT / "runtime/model_selection/card/release16_rulebase_reversal_drawdown_upside_checkpoint_metric_replay_v1/selection.json"
STATE_HEAD_METRICS = ROOT / "runtime/model_selection/card/release16_state_head_post_fullprobe_objective_screen_v1/walkforward_candidate_metrics.csv"
SOXLBACKTEST = ROOT / "runtime/model_selection/card/soxlbacktest_qlib_walkforward_baseline_v1/candidate_metrics.csv"
REFERENCE_SOURCE_LABELS = {"soxlbacktest_qlib_walkforward_baseline_v1"}
RETENTION_GATE = ROOT / "runtime/model_selection/card/model_retention_walkforward_gate_v1/summary.json"


def json_safe(value: Any) -> Any:
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.bool_, bool)):
        return bool(value)
    if isinstance(value, (np.floating,)):
        value = float(value)
    if isinstance(value, float):
        return value if math.isfinite(value) else None
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    if isinstance(value, Path):
        return str(value)
    return value


def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.write_text(json.dumps(json_safe(payload), indent=2, sort_keys=True, allow_nan=False) + "\n", encoding="utf-8")


def load_json(path: Path) -> dict[str, Any]:
    return json.loads(path.read_text(encoding="utf-8")) if path.exists() else {}

print("root", ROOT)
print("card", CARD)


root /Users/rziiiii/dev/workspace/.worktrees/dashboard-v2-model-features-fix
card /Users/rziiiii/dev/workspace/.worktrees/dashboard-v2-model-features-fix/runtime/model_selection/card/release16_official_replay_candidate_handoff_v1


In [2]:

frames: list[pd.DataFrame] = []
source_rows: list[dict[str, Any]] = []

current = pd.read_csv(CURRENT_REPLAY)
current = current[current["fold"].isin(WINDOWS)].copy()
frames.append(
    normalise_official_replay_candidate_frame(
        current,
        source_label="release_gate_bc_official_replay_v1",
        mechanism="release_gate_bc_weighting",
        candidate_col="candidate",
        window_col="fold",
        relative_col="relative_return_pp",
        max_dd_col="max_dd_pct",
        avg_pos_col="avg_pos",
        config=PRIMARY_CONFIG,
    )
)
source_rows.append({"source": "current_release_gate_bc", "path": str(CURRENT_REPLAY.relative_to(ROOT)), "rows": int(len(current))})

prev = pd.read_csv(PREV_CHECKPOINT)
prev = prev[prev["fold"].isin(WINDOWS)].copy()
frames.append(
    normalise_official_replay_candidate_frame(
        prev,
        source_label="release16_drawdown_upside_checkpoint_replay_v1",
        mechanism="checkpoint_metric_replay",
        candidate_col="candidate",
        window_col="fold",
        relative_col="test_relative_return",
        max_dd_col="test_max_dd",
        avg_pos_col="test_avg_pos",
        gate_col="candidate_gate",
        config=PRIMARY_CONFIG,
    )
)
source_rows.append({"source": "previous_release16_checkpoint", "path": str(PREV_CHECKPOINT.relative_to(ROOT)), "rows": int(len(prev))})

if STATE_HEAD_METRICS.exists():
    state_head = pd.read_csv(STATE_HEAD_METRICS)
    state_head = state_head[state_head["window"].isin(WINDOWS)].copy()
    frames.append(
        normalise_official_replay_candidate_frame(
            state_head,
            source_label="release16_state_head_post_fullprobe_objective_screen_v1",
            mechanism="state_head_objective_screen",
            candidate_col="candidate",
            window_col="window",
            relative_col="relative_return_pp",
            max_dd_col="max_dd_pct",
            avg_pos_col="avg_position",
            gate_col="fold_gate",
            config=PRIMARY_CONFIG,
        )
    )
    source_rows.append({"source": "state_head_objective_screen", "path": str(STATE_HEAD_METRICS.relative_to(ROOT)), "rows": int(len(state_head))})

soxl = pd.read_csv(SOXLBACKTEST)
soxl = soxl[soxl["window"].isin(WINDOWS) & soxl["split_type"].eq("walk_forward_test")].copy()
soxl["relative_return_pp"] = pd.to_numeric(soxl["rulebase_return_pct"], errors="coerce") - pd.to_numeric(soxl["soxl_buyhold_return_pct"], errors="coerce")
soxl["avg_pos"] = pd.to_numeric(soxl["average_position_pct"], errors="coerce") / 100.0
soxl = soxl[soxl["candidate"].isin(["legacy_default", "wf_pass_ytd18_15_15_02", "risk_cap_35_15_05", "low_risk_20_08_02", "low_risk_15_05_02"])].copy()
frames.append(
    normalise_official_replay_candidate_frame(
        soxl,
        source_label="soxlbacktest_qlib_walkforward_baseline_v1",
        mechanism="qlib_converted_buysell_reference",
        candidate_col="candidate",
        window_col="window",
        relative_col="relative_return_pp",
        max_dd_col="rulebase_max_drawdown_pct",
        avg_pos_col="avg_pos",
        config=PRIMARY_CONFIG,
    )
)
source_rows.append({"source": "soxlbacktest_qlib_reference", "path": str(SOXLBACKTEST.relative_to(ROOT)), "rows": int(len(soxl))})

candidate_windows = pd.concat(frames, ignore_index=True)
candidate_windows = candidate_windows[candidate_windows["window"].isin(WINDOWS)].copy()
# Keep only candidates with complete three-fold coverage so ranking is apples-to-apples.
coverage = candidate_windows.groupby("candidate_id")["window"].nunique()
complete_ids = set(coverage[coverage == len(WINDOWS)].index.astype(str))
candidate_windows = candidate_windows[candidate_windows["candidate_id"].astype(str).isin(complete_ids)].reset_index(drop=True)

# Recompute primary and strict gates explicitly for clarity.
candidate_windows["primary_window_gate"] = (candidate_windows["relative_return"] >= 30.0) & (candidate_windows["max_dd"] <= 20.0)
candidate_windows["strict_window_gate"] = (candidate_windows["relative_return"] >= 30.0) & (candidate_windows["max_dd"] <= 15.0)
candidate_windows["relative_gap_20dd"] = (30.0 - candidate_windows["relative_return"]).clip(lower=0.0)
candidate_windows["max_dd_gap_20dd"] = (candidate_windows["max_dd"] - 20.0).clip(lower=0.0)
candidate_windows["relative_gap_15dd"] = (30.0 - candidate_windows["relative_return"]).clip(lower=0.0)
candidate_windows["max_dd_gap_15dd"] = (candidate_windows["max_dd"] - 15.0).clip(lower=0.0)

candidate_windows.to_csv(CARD / "candidate_window_metrics.csv", index=False)
pd.DataFrame(source_rows).to_csv(CARD / "source_tables.csv", index=False)
display(pd.DataFrame(source_rows))
display(candidate_windows.head(20))


,source,path,rows
0,current_release_gate_bc,runtime/model_selection/card/release16_release...,6
1,previous_release16_checkpoint,runtime/model_selection/card/release16_rulebas...,12
2,state_head_objective_screen,runtime/model_selection/card/release16_state_h...,15
3,soxlbacktest_qlib_reference,runtime/model_selection/card/soxlbacktest_qlib...,15


,source_label,mechanism,candidate,candidate_id,window,relative_return,max_dd,avg_pos,window_gate_pass,relative_gap,max_dd_gap,robust_window_gate,primary_window_gate,strict_window_gate,relative_gap_20dd,max_dd_gap_20dd,relative_gap_15dd,max_dd_gap_15dd
0,release_gate_bc_official_replay_v1,release_gate_bc_weighting,baseline_current,release_gate_bc_official_replay_v1::baseline_c...,wf_2020,15.901974,24.834087,0.396934,False,14.098026,4.834087,False,False,False,14.098026,4.834087,14.098026,9.834087
1,release_gate_bc_official_replay_v1,release_gate_bc_weighting,release64_no_objective,release_gate_bc_official_replay_v1::release64_...,wf_2020,22.281639,21.106156,0.425202,False,7.718361,1.106156,False,False,False,7.718361,1.106156,7.718361,6.106156
2,release_gate_bc_official_replay_v1,release_gate_bc_weighting,baseline_current,release_gate_bc_official_replay_v1::baseline_c...,wf_2022,70.560022,20.401262,0.093902,False,0.000000,0.401262,False,False,False,0.000000,0.401262,0.000000,5.401262
3,release_gate_bc_official_replay_v1,release_gate_bc_weighting,release64_no_objective,release_gate_bc_official_replay_v1::release64_...,wf_2022,62.750639,26.860565,0.144199,False,0.000000,6.860565,False,False,False,0.000000,6.860565,0.000000,11.860565
4,release_gate_bc_official_replay_v1,release_gate_bc_weighting,baseline_current,release_gate_bc_official_replay_v1::baseline_c...,wf_2023_2024,7.186027,13.622379,0.224851,False,22.813973,0.000000,False,False,False,22.813973,0.000000,22.813973,0.000000
5,release_gate_bc_official_replay_v1,release_gate_bc_weighting,release64_no_objective,release_gate_bc_official_replay_v1::release64_...,wf_2023_2024,14.302062,13.959434,0.197766,False,15.697938,0.000000,False,False,False,15.697938,0.000000,15.697938,0.000000
6,release16_drawdown_upside_checkpoint_replay_v1,checkpoint_metric_replay,bc_initialization,release16_drawdown_upside_checkpoint_replay_v1...,wf_2020,-0.427053,19.326906,0.302925,False,30.427053,0.000000,False,False,False,30.427053,0.000000,30.427053,4.326906
7,release16_drawdown_upside_checkpoint_replay_v1,checkpoint_metric_replay,ppo_retained_iter_0_bc_equivalent,release16_drawdown_upside_checkpoint_replay_v1...,wf_2020,-0.427053,19.326906,0.302925,False,30.427053,0.000000,False,False,False,30.427053,0.000000,30.427053,4.326906
8,release16_drawdown_upside_checkpoint_replay_v1,checkpoint_metric_replay,ppo_retained_iter_4,release16_drawdown_upside_checkpoint_replay_v1...,wf_2020,-13.714574,25.585056,0.418305,False,43.714574,5.585056,False,False,False,43.714574,5.585056,43.714574,10.585056
9,release16_drawdown_upside_checkpoint_replay_v1,checkpoint_metric_replay,ppo_selected_best,release16_drawdown_upside_checkpoint_replay_v1...,wf_2020,-13.714574,25.585056,0.418305,False,43.714574,5.585056,False,False,False,43.714574,5.585056,43.714574,10.585056


In [3]:

primary_scores = score_official_replay_candidate_pool(candidate_windows, config=PRIMARY_CONFIG)
strict_frames = []
for source_label, sub in candidate_windows.groupby("source_label"):
    strict_frames.append(
        normalise_official_replay_candidate_frame(
            sub.rename(columns={"window": "fold"}),
            source_label=str(source_label),
            mechanism=str(sub["mechanism"].iloc[0]),
            candidate_col="candidate",
            window_col="fold",
            relative_col="relative_return",
            max_dd_col="max_dd",
            avg_pos_col="avg_pos",
            gate_col="strict_window_gate",
            config=STRICT_CONFIG,
        )
    )
strict_windows = pd.concat(strict_frames, ignore_index=True)
strict_scores = score_official_replay_candidate_pool(strict_windows, config=STRICT_CONFIG)

stress = score_official_replay_stress_scenarios(
    candidate_windows,
    stress_scenarios=[
        {"scenario": "relative_minus_1pp", "relative_haircut_pp": 1.0, "max_dd_add_pct": 0.0},
        {"scenario": "dd_plus_1pp", "relative_haircut_pp": 0.0, "max_dd_add_pct": 1.0},
        {"scenario": "relative_minus_1_dd_plus_1", "relative_haircut_pp": 1.0, "max_dd_add_pct": 1.0},
    ],
    config=PRIMARY_CONFIG,
)

# Add worst-window attribution for the primary score table.
worst_rows = []
for candidate_id, sub in candidate_windows.groupby("candidate_id"):
    work = sub.copy()
    work["gate_loss_20dd"] = work["relative_gap_20dd"] + work["max_dd_gap_20dd"] * 10.0
    worst = work.sort_values(["primary_window_gate", "gate_loss_20dd", "relative_gap_20dd", "max_dd_gap_20dd"], ascending=[True, False, False, False]).iloc[0]
    rel_bad = float(worst["relative_gap_20dd"]) > 0.0
    dd_bad = float(worst["max_dd_gap_20dd"]) > 0.0
    if rel_bad and dd_bad:
        fold_failure = "joint_tradeoff"
    elif rel_bad:
        fold_failure = "trend_capture"
    elif dd_bad:
        fold_failure = "drawdown_control"
    else:
        fold_failure = "passed"
    worst_rows.append({
        "candidate_id": candidate_id,
        "worst_window": worst["window"],
        "worst_window_failure_type": fold_failure,
        "worst_window_relative_return": float(worst["relative_return"]),
        "worst_window_max_dd": float(worst["max_dd"]),
        "worst_window_relative_gap": float(worst["relative_gap_20dd"]),
        "worst_window_max_dd_gap": float(worst["max_dd_gap_20dd"]),
        "worst_window_gate_loss": float(worst["gate_loss_20dd"]),
    })
worst_attribution = pd.DataFrame(worst_rows)
primary_scores = primary_scores.merge(worst_attribution, on="candidate_id", how="left")
primary_scores["weighted_objective_loss"] = primary_scores["total_gate_loss"]

primary_scores.to_csv(CARD / "official_candidate_scores_20dd.csv", index=False)
strict_scores.to_csv(CARD / "official_candidate_scores_15dd.csv", index=False)
stress.to_csv(CARD / "official_candidate_stress_scores.csv", index=False)
worst_attribution.to_csv(CARD / "worst_window_attribution.csv", index=False)

display(primary_scores)
display(stress.head(20))


,candidate_id,source_label,mechanism,candidate,window_count,gate_pass_count,full_gate,robust_gate_pass_count,robust_full_gate,near_miss,...,total_gate_loss,failure_profile,worst_window,worst_window_failure_type,worst_window_relative_return,worst_window_max_dd,worst_window_relative_gap,worst_window_max_dd_gap,worst_window_gate_loss,weighted_objective_loss
0,release16_drawdown_upside_checkpoint_replay_v1...,release16_drawdown_upside_checkpoint_replay_v1,checkpoint_metric_replay,bc_initialization,3,1,False,1,False,False,...,57.435335,trend_capture,wf_2020,trend_capture,-0.427053,19.326906,30.427053,0.000000,30.427053,57.435335
1,release16_drawdown_upside_checkpoint_replay_v1...,release16_drawdown_upside_checkpoint_replay_v1,checkpoint_metric_replay,ppo_retained_iter_0_bc_equivalent,3,1,False,1,False,False,...,57.435335,trend_capture,wf_2020,trend_capture,-0.427053,19.326906,30.427053,0.000000,30.427053,57.435335
2,release16_state_head_post_fullprobe_objective_...,release16_state_head_post_fullprobe_objective_...,state_head_objective_screen,checkpoint_bc_initialization,3,0,False,1,False,False,...,57.435335,trend_capture,wf_2020,trend_capture,-0.427053,19.326906,30.427053,0.000000,30.427053,57.435335
3,release16_state_head_post_fullprobe_objective_...,release16_state_head_post_fullprobe_objective_...,state_head_objective_screen,checkpoint_ppo_retained_iter_0_bc_equivalent,3,0,False,1,False,False,...,57.435335,trend_capture,wf_2020,trend_capture,-0.427053,19.326906,30.427053,0.000000,30.427053,57.435335
4,soxlbacktest_qlib_walkforward_baseline_v1::wf_...,soxlbacktest_qlib_walkforward_baseline_v1,qlib_converted_buysell_reference,wf_pass_ytd18_15_15_02,3,1,False,1,False,False,...,83.509497,trend_capture,wf_2020,trend_capture,-30.618705,12.785207,60.618705,0.000000,60.618705,83.509497
5,soxlbacktest_qlib_walkforward_baseline_v1::low...,soxlbacktest_qlib_walkforward_baseline_v1,qlib_converted_buysell_reference,low_risk_15_05_02,3,1,False,0,False,False,...,88.256963,trend_capture,wf_2020,trend_capture,-32.131801,6.828971,62.131801,0.000000,62.131801,88.256963
6,release_gate_bc_official_replay_v1::baseline_c...,release_gate_bc_official_replay_v1,release_gate_bc_weighting,baseline_current,3,0,False,0,False,False,...,89.265484,joint_tradeoff,wf_2020,joint_tradeoff,15.901974,24.834087,14.098026,4.834087,62.438895,89.265484
7,release16_state_head_post_fullprobe_objective_...,release16_state_head_post_fullprobe_objective_...,state_head_objective_screen,full_actor_probe,3,0,False,1,False,False,...,99.196903,joint_tradeoff,wf_2020,trend_capture,-22.718827,19.260466,52.718827,0.000000,52.718827,99.196903
8,release_gate_bc_official_replay_v1::release64_...,release_gate_bc_official_replay_v1,release_gate_bc_weighting,release64_no_objective,3,0,False,0,False,False,...,103.083505,joint_tradeoff,wf_2022,drawdown_control,62.750639,26.860565,0.000000,6.860565,68.605646,103.083505
9,soxlbacktest_qlib_walkforward_baseline_v1::low...,soxlbacktest_qlib_walkforward_baseline_v1,qlib_converted_buysell_reference,low_risk_20_08_02,3,0,False,0,False,False,...,126.786339,joint_tradeoff,wf_2020,trend_capture,-24.165599,9.073791,54.165599,0.000000,54.165599,126.786339


,candidate_id,source_label,mechanism,candidate,scenario,relative_haircut_pp,max_dd_add_pct,window_count,stress_gate_pass_count,stress_full_gate,stress_min_relative_return,stress_max_dd,stress_total_gate_loss,stress_failure_profile
0,release16_drawdown_upside_checkpoint_replay_v1...,release16_drawdown_upside_checkpoint_replay_v1,checkpoint_metric_replay,bc_initialization,dd_plus_1pp,0.0,1.0,3,1,False,-0.427053,20.326906,60.704399,stress_trend_capture
1,release16_drawdown_upside_checkpoint_replay_v1...,release16_drawdown_upside_checkpoint_replay_v1,checkpoint_metric_replay,ppo_retained_iter_0_bc_equivalent,dd_plus_1pp,0.0,1.0,3,1,False,-0.427053,20.326906,60.704399,stress_trend_capture
2,release16_state_head_post_fullprobe_objective_...,release16_state_head_post_fullprobe_objective_...,state_head_objective_screen,checkpoint_bc_initialization,dd_plus_1pp,0.0,1.0,3,1,False,-0.427053,20.326906,60.704399,stress_trend_capture
3,release16_state_head_post_fullprobe_objective_...,release16_state_head_post_fullprobe_objective_...,state_head_objective_screen,checkpoint_ppo_retained_iter_0_bc_equivalent,dd_plus_1pp,0.0,1.0,3,1,False,-0.427053,20.326906,60.704399,stress_trend_capture
4,soxlbacktest_qlib_walkforward_baseline_v1::wf_...,soxlbacktest_qlib_walkforward_baseline_v1,qlib_converted_buysell_reference,wf_pass_ytd18_15_15_02,dd_plus_1pp,0.0,1.0,3,1,False,-30.618705,19.514585,83.509497,stress_trend_capture
5,soxlbacktest_qlib_walkforward_baseline_v1::low...,soxlbacktest_qlib_walkforward_baseline_v1,qlib_converted_buysell_reference,low_risk_15_05_02,dd_plus_1pp,0.0,1.0,3,0,False,-32.131801,20.456050,92.817463,stress_trend_capture
6,release_gate_bc_official_replay_v1::baseline_c...,release_gate_bc_official_replay_v1,release_gate_bc_weighting,baseline_current,dd_plus_1pp,0.0,1.0,3,0,False,7.186027,25.834087,109.265484,stress_joint_tradeoff
7,release16_state_head_post_fullprobe_objective_...,release16_state_head_post_fullprobe_objective_...,state_head_objective_screen,full_actor_probe,dd_plus_1pp,0.0,1.0,3,1,False,-22.718827,21.914378,111.801564,stress_joint_tradeoff
8,release_gate_bc_official_replay_v1::release64_...,release_gate_bc_official_replay_v1,release_gate_bc_weighting,release64_no_objective,dd_plus_1pp,0.0,1.0,3,0,False,14.302062,27.860565,123.083505,stress_joint_tradeoff
9,soxlbacktest_qlib_walkforward_baseline_v1::low...,soxlbacktest_qlib_walkforward_baseline_v1,qlib_converted_buysell_reference,low_risk_20_08_02,dd_plus_1pp,0.0,1.0,3,0,False,-24.165599,25.732060,136.786339,stress_joint_tradeoff


In [4]:

# Fold-level gate gap attribution for the leading candidates.
lead_ids = primary_scores.head(6)["candidate_id"].astype(str).tolist()
gate_gap_rows = []
for _, row in candidate_windows[candidate_windows["candidate_id"].isin(lead_ids)].iterrows():
    rel_gap = float(row["relative_gap_20dd"])
    dd_gap = float(row["max_dd_gap_20dd"])
    if rel_gap > 0 and dd_gap > 0:
        failure_type = "joint_tradeoff"
    elif rel_gap > 0:
        failure_type = "underparticipation"
    elif dd_gap > 0:
        failure_type = "drawdown_control"
    else:
        failure_type = "passed"
    gate_gap_rows.append({
        "candidate_id": row["candidate_id"],
        "source_label": row["source_label"],
        "candidate": row["candidate"],
        "window": row["window"],
        "relative_return": float(row["relative_return"]),
        "max_dd": float(row["max_dd"]),
        "avg_pos": float(row["avg_pos"]) if pd.notna(row["avg_pos"]) else None,
        "relative_gap": rel_gap,
        "max_dd_gap": dd_gap,
        "failure_type": failure_type,
    })
gate_gap_attribution = pd.DataFrame(gate_gap_rows)
gate_gap_profile = (
    gate_gap_attribution.groupby(["candidate_id", "failure_type"], dropna=False)
    .size()
    .reset_index(name="window_count")
    .sort_values(["candidate_id", "window_count"], ascending=[True, False])
)

# Use non-reference relative-gap windows as a lightweight underparticipation proxy for the handoff helper.
# SOXLBacktest rows remain in the pool as reference baselines, not trainable objective drivers.
trainable_gate_gap_attribution = gate_gap_attribution[~gate_gap_attribution["source_label"].isin(REFERENCE_SOURCE_LABELS)].copy()
poor_proxy = trainable_gate_gap_attribution[trainable_gate_gap_attribution["relative_gap"] > 0].copy()
poor_proxy["dominant_failure_type"] = "underparticipation"
poor_proxy["mean_excess_return"] = poor_proxy["relative_return"]
poor_proxy = poor_proxy.rename(columns={"window": "cluster_id", "relative_gap": "mean_relative_gap"})
poor_proxy = poor_proxy[["candidate_id", "cluster_id", "dominant_failure_type", "mean_excess_return", "mean_relative_gap", "max_dd_gap"]]

# Build feature group hints from the selected candidate failure types; these are mechanism labels, not factor claims.
feature_group_rows = []
for failure_type, groups in {
    "drawdown_control": ["drawdown_path", "volatility_regime", "sequence_risk"],
    "underparticipation": ["trend_momentum", "relative_strength_spread", "release_rebound"],
    "joint_tradeoff": ["drawdown_path", "trend_momentum", "volatility_regime"],
}.items():
    count = int((trainable_gate_gap_attribution["failure_type"] == failure_type).sum())
    for group in groups:
        feature_group_rows.append({"failure_type": failure_type, "feature_group": group, "top_mentions": count, "mean_abs_delta_z": float(count)})
feature_group_consensus = pd.DataFrame(feature_group_rows)

# Handoff operates on candidate-level scores. It expects a candidate column and gate/loss fields.
handoff_scores = primary_scores[~primary_scores["source_label"].isin(REFERENCE_SOURCE_LABELS)].rename(columns={"full_gate": "objective_pass", "total_gate_loss": "objective_loss"}).copy()
handoff_scores["candidate"] = handoff_scores["candidate_id"]
handoff_scores["drawdown_loss"] = handoff_scores["max_dd_gap"]
handoff_scores["relative_loss"] = handoff_scores["max_relative_gap"]
handoff_scores["participation_loss"] = handoff_scores["max_relative_gap"]
handoff_scores["dominant_failure_type"] = handoff_scores["failure_profile"].replace({"trend_capture": "underparticipation"})

handoff = derive_training_objective_handoff(
    walkforward_scores=handoff_scores,
    poor_window_clusters=poor_proxy,
    feature_groups=feature_group_consensus,
    router_selection=load_json(CURRENT_SELECTION),
)
(CARD / "training_objective_handoff.md").write_text(render_training_objective_handoff(handoff) + "\n", encoding="utf-8")
write_json(CARD / "training_objective_handoff.json", handoff)

gate_gap_attribution.to_csv(CARD / "gate_gap_attribution.csv", index=False)
gate_gap_profile.to_csv(CARD / "gate_gap_profile_summary.csv", index=False)
poor_proxy.to_csv(CARD / "underparticipation_gap_proxy.csv", index=False)
feature_group_consensus.to_csv(CARD / "feature_group_consensus.csv", index=False)

display(gate_gap_attribution)
display(gate_gap_profile)
print(json.dumps(json_safe(handoff), indent=2, sort_keys=True))


,candidate_id,source_label,candidate,window,relative_return,max_dd,avg_pos,relative_gap,max_dd_gap,failure_type
0,release16_drawdown_upside_checkpoint_replay_v1...,release16_drawdown_upside_checkpoint_replay_v1,bc_initialization,wf_2020,-0.427053,19.326906,0.302925,30.427053,0.0,underparticipation
1,release16_drawdown_upside_checkpoint_replay_v1...,release16_drawdown_upside_checkpoint_replay_v1,ppo_retained_iter_0_bc_equivalent,wf_2020,-0.427053,19.326906,0.302925,30.427053,0.0,underparticipation
2,release16_drawdown_upside_checkpoint_replay_v1...,release16_drawdown_upside_checkpoint_replay_v1,bc_initialization,wf_2022,71.714980,15.617043,0.081384,0.000000,0.0,passed
3,release16_drawdown_upside_checkpoint_replay_v1...,release16_drawdown_upside_checkpoint_replay_v1,ppo_retained_iter_0_bc_equivalent,wf_2022,71.714980,15.617043,0.081384,0.000000,0.0,passed
4,release16_drawdown_upside_checkpoint_replay_v1...,release16_drawdown_upside_checkpoint_replay_v1,bc_initialization,wf_2023_2024,2.991718,13.624617,0.216822,27.008282,0.0,underparticipation
5,release16_drawdown_upside_checkpoint_replay_v1...,release16_drawdown_upside_checkpoint_replay_v1,ppo_retained_iter_0_bc_equivalent,wf_2023_2024,2.991718,13.624617,0.216822,27.008282,0.0,underparticipation
6,release16_state_head_post_fullprobe_objective_...,release16_state_head_post_fullprobe_objective_...,checkpoint_bc_initialization,wf_2020,-0.427053,19.326906,0.302925,30.427053,0.0,underparticipation
7,release16_state_head_post_fullprobe_objective_...,release16_state_head_post_fullprobe_objective_...,checkpoint_bc_initialization,wf_2022,71.714980,15.617043,0.081384,0.000000,0.0,passed
8,release16_state_head_post_fullprobe_objective_...,release16_state_head_post_fullprobe_objective_...,checkpoint_bc_initialization,wf_2023_2024,2.991718,13.624617,0.216822,27.008282,0.0,underparticipation
9,release16_state_head_post_fullprobe_objective_...,release16_state_head_post_fullprobe_objective_...,checkpoint_ppo_retained_iter_0_bc_equivalent,wf_2020,-0.427053,19.326906,0.302925,30.427053,0.0,underparticipation


,candidate_id,failure_type,window_count
1,release16_drawdown_upside_checkpoint_replay_v1...,underparticipation,2
0,release16_drawdown_upside_checkpoint_replay_v1...,passed,1
3,release16_drawdown_upside_checkpoint_replay_v1...,underparticipation,2
2,release16_drawdown_upside_checkpoint_replay_v1...,passed,1
5,release16_state_head_post_fullprobe_objective_...,underparticipation,2
4,release16_state_head_post_fullprobe_objective_...,passed,1
7,release16_state_head_post_fullprobe_objective_...,underparticipation,2
6,release16_state_head_post_fullprobe_objective_...,passed,1
9,soxlbacktest_qlib_walkforward_baseline_v1::low...,underparticipation,2
8,soxlbacktest_qlib_walkforward_baseline_v1::low...,passed,1


{
  "decision": "train_rebound_participation_objective",
  "input_counts": {
    "feature_groups": 9,
    "poor_window_clusters": 8,
    "walkforward_candidates": 11
  },
  "objective_terms": [
    "walkforward_relative_return_gap_penalty",
    "walkforward_drawdown_gap_penalty",
    "high_vol_rebound_participation_floor"
  ],
  "primary_underparticipation_cluster": {
    "candidate_id": "release16_drawdown_upside_checkpoint_replay_v1::bc_initialization",
    "cluster_id": "wf_2020",
    "dominant_failure_type": "underparticipation",
    "max_dd_gap": 0.0,
    "mean_excess_return": -0.4270525146105939,
    "mean_relative_gap": 30.427052514610594
  },
  "promotion": "blocked",
  "risk_gap": {
    "max_dd_gap": 4.33,
    "observed_max_dd": 19.33,
    "observed_min_relative_return": -0.43,
    "relative_gap": 30.43,
    "target_max_dd": 15.0,
    "target_relative_return": 30.0,
    "worst_window": "wf_2020"
  },
  "router_cluster_delta_max": 0.0,
  "selected_diagnostic_candidate": "releas

In [5]:

current_selection = load_json(CURRENT_SELECTION)
weight_summary = load_json(WEIGHT_SCREEN)
retention_summary = load_json(RETENTION_GATE).get("summary", {})
best = primary_scores.iloc[0].to_dict() if not primary_scores.empty else {}
best_strict = strict_scores.iloc[0].to_dict() if not strict_scores.empty else {}
full_gate_count = int(primary_scores["full_gate"].fillna(False).astype(bool).sum()) if not primary_scores.empty else 0
strict_full_gate_count = int(strict_scores["full_gate"].fillna(False).astype(bool).sum()) if not strict_scores.empty else 0
trainable_primary_scores = primary_scores[~primary_scores["source_label"].isin(REFERENCE_SOURCE_LABELS)].copy()
reference_primary_scores = primary_scores[primary_scores["source_label"].isin(REFERENCE_SOURCE_LABELS)].copy()
trainable_candidate_count = int(trainable_primary_scores["candidate_id"].nunique()) if not trainable_primary_scores.empty else 0
reference_candidate_count = int(reference_primary_scores["candidate_id"].nunique()) if not reference_primary_scores.empty else 0
nonreference_full_gate_count = int(trainable_primary_scores["full_gate"].fillna(False).astype(bool).sum()) if not trainable_primary_scores.empty else 0
reference_full_gate_count = int(reference_primary_scores["full_gate"].fillna(False).astype(bool).sum()) if not reference_primary_scores.empty else 0
best_is_reference = str(best.get("source_label", "")).startswith("soxlbacktest")

if nonreference_full_gate_count > 0:
    decision = "official_candidate_pool_has_nonreference_full_gate_needs_retention_review"
    next_mechanism = "run_fixed_split_ytd_dashboard_replay_and_actor_retention_gate"
elif reference_full_gate_count > 0:
    decision = "official_candidate_pool_reference_only_full_gate_no_actor_promotion"
    next_mechanism = "translate_reference_behavior_into_trainable_checkpoint_objective"
else:
    decision = "official_candidate_pool_no_full_gate_training_objective_handoff"
    next_mechanism = str(handoff.get("decision", "train_walkforward_weighted_objective"))

selection = {
    "experiment": EXPERIMENT,
    "decision": decision,
    "next_mechanism": next_mechanism,
    "best_candidate_id": best.get("candidate_id"),
    "best_failure_profile": best.get("failure_profile"),
    "best_total_gate_loss": best.get("total_gate_loss"),
    "best_worst_window": best.get("worst_window"),
    "full_gate_count_20dd": full_gate_count,
    "full_gate_count_15dd": strict_full_gate_count,
    "nonreference_full_gate_count_20dd": nonreference_full_gate_count,
    "reference_full_gate_count_20dd": reference_full_gate_count,
    "promotion_allowed": False,
    "actor_retention_allowed": False,
}
write_json(CARD / "selection.json", selection)

summary = {
    "experiment": EXPERIMENT,
    "created_at": datetime.now(timezone.utc).isoformat(),
    "decision": decision,
    "promotion_allowed": False,
    "actor_retention_allowed": False,
    "role": "diagnose_official_candidate_handoff",
    "canonical_baseline": "soxlbacktest_policy_reimplementation_v1",
    "direct_comparison_baseline": "release16_release_gate_bc_official_replay_v1",
    "target_window": WINDOWS,
    "changed_layer": ["model_selection", "checkpoint_objective", "training_objective_handoff"],
    "external_reference": "soxlbacktest_qlib_walkforward_baseline_v1 and release16 official replay candidates",
    "hypothesis": "A unified official replay candidate pool will distinguish drawdown-control, underparticipation, and joint-tradeoff blockers after release-gate BC slice repair.",
    "acceptance_criteria": [
        "candidate pool uses official qlib daily path metrics where available",
        "primary gate uses relative_return >= 30pp and MaxDD <= 20% across all walk-forward folds",
        "strict <=15% MaxDD scores are retained for risk context",
        "no candidate is promoted without fixed/YTD/dashboard/retention evidence",
    ],
    "expected_tradeoff": "Release-gate weighting improved supervised fit and some return windows, but official replay may require a joint objective that penalizes drawdown drift and relative-return gaps simultaneously.",
    "failure_mode": "A low-loss candidate can still be reference-only, stress-fragile, or fail one fold badly enough to block PPO and actor retention.",
    "comparison_table_row_key": EXPERIMENT,
    "closest_historical_success_case": weight_summary.get("closest_historical_success_case", "release16_release_gate_bc_weight_screen_v1 fixed release-gate slice fit"),
    "closest_historical_failure_case": current_selection.get("decision", "release_gate_weighted_bc_official_replay_failed_no_promotion"),
    "why_not_duplicate": "This aggregates multiple official replay sources and emits a handoff objective; prior notebooks only screened one weighting config or one candidate pair.",
    "primary_gate": {"relative_return_pp": 30.0, "max_dd_pct": 20.0},
    "strict_gate": {"relative_return_pp": 30.0, "max_dd_pct": 15.0},
    "pool_summary": {
        "candidate_window_rows": int(len(candidate_windows)),
        "candidate_count": int(primary_scores["candidate_id"].nunique()) if not primary_scores.empty else 0,
        "trainable_candidate_count": trainable_candidate_count,
        "reference_candidate_count": reference_candidate_count,
        "full_gate_count_20dd": full_gate_count,
        "nonreference_full_gate_count_20dd": nonreference_full_gate_count,
        "reference_full_gate_count_20dd": reference_full_gate_count,
        "full_gate_count_15dd": strict_full_gate_count,
        "best_candidate_id": best.get("candidate_id"),
        "best_source_label": best.get("source_label"),
        "best_failure_profile": best.get("failure_profile"),
        "best_total_gate_loss": best.get("total_gate_loss"),
        "best_worst_window": best.get("worst_window"),
        "best_worst_window_failure_type": best.get("worst_window_failure_type"),
        "best_min_relative_return": best.get("min_relative_return"),
        "best_max_max_dd": best.get("max_max_dd"),
    },
    "training_objective_handoff": handoff,
    "retention_gate_current": {
        "passing_profile_count": retention_summary.get("passing_profile_count"),
        "reference_profile": retention_summary.get("reference_profile"),
        "reference_return_pct": retention_summary.get("reference_return_pct"),
        "current_best_return_pct": retention_summary.get("current_best_return_pct"),
        "required_best_return_pct": retention_summary.get("required_best_return_pct"),
        "max_walkforward_drawdown_pct": retention_summary.get("max_walkforward_drawdown_pct"),
    },
    "output_paths": {
        "candidate_window_metrics": str((CARD / "candidate_window_metrics.csv").relative_to(ROOT)),
        "official_candidate_scores_20dd": str((CARD / "official_candidate_scores_20dd.csv").relative_to(ROOT)),
        "official_candidate_scores_15dd": str((CARD / "official_candidate_scores_15dd.csv").relative_to(ROOT)),
        "official_candidate_stress_scores": str((CARD / "official_candidate_stress_scores.csv").relative_to(ROOT)),
        "gate_gap_attribution": str((CARD / "gate_gap_attribution.csv").relative_to(ROOT)),
        "gate_gap_profile_summary": str((CARD / "gate_gap_profile_summary.csv").relative_to(ROOT)),
        "training_objective_handoff": str((CARD / "training_objective_handoff.json").relative_to(ROOT)),
        "training_objective_handoff_markdown": str((CARD / "training_objective_handoff.md").relative_to(ROOT)),
        "selection": str((CARD / "selection.json").relative_to(ROOT)),
        "summary": str((CARD / "summary.json").relative_to(ROOT)),
    },
    "next_mechanism": next_mechanism,
    "conclusion": (
        "A non-reference candidate passed the primary official replay gate; still no promotion until retention review surfaces pass."
        if decision == "official_candidate_pool_has_nonreference_full_gate_needs_retention_review"
        else "No trainable non-reference candidate passes the official walk-forward pool gate; next step is the emitted training/checkpoint objective handoff, not PPO length or actor retention."
    ),
}
write_json(CARD / "summary.json", summary)

print("decision", decision)
print("next", next_mechanism)
display(primary_scores.head(10))


decision official_candidate_pool_no_full_gate_training_objective_handoff
next train_rebound_participation_objective


,candidate_id,source_label,mechanism,candidate,window_count,gate_pass_count,full_gate,robust_gate_pass_count,robust_full_gate,near_miss,...,total_gate_loss,failure_profile,worst_window,worst_window_failure_type,worst_window_relative_return,worst_window_max_dd,worst_window_relative_gap,worst_window_max_dd_gap,worst_window_gate_loss,weighted_objective_loss
0,release16_drawdown_upside_checkpoint_replay_v1...,release16_drawdown_upside_checkpoint_replay_v1,checkpoint_metric_replay,bc_initialization,3,1,False,1,False,False,...,57.435335,trend_capture,wf_2020,trend_capture,-0.427053,19.326906,30.427053,0.000000,30.427053,57.435335
1,release16_drawdown_upside_checkpoint_replay_v1...,release16_drawdown_upside_checkpoint_replay_v1,checkpoint_metric_replay,ppo_retained_iter_0_bc_equivalent,3,1,False,1,False,False,...,57.435335,trend_capture,wf_2020,trend_capture,-0.427053,19.326906,30.427053,0.000000,30.427053,57.435335
2,release16_state_head_post_fullprobe_objective_...,release16_state_head_post_fullprobe_objective_...,state_head_objective_screen,checkpoint_bc_initialization,3,0,False,1,False,False,...,57.435335,trend_capture,wf_2020,trend_capture,-0.427053,19.326906,30.427053,0.000000,30.427053,57.435335
3,release16_state_head_post_fullprobe_objective_...,release16_state_head_post_fullprobe_objective_...,state_head_objective_screen,checkpoint_ppo_retained_iter_0_bc_equivalent,3,0,False,1,False,False,...,57.435335,trend_capture,wf_2020,trend_capture,-0.427053,19.326906,30.427053,0.000000,30.427053,57.435335
4,soxlbacktest_qlib_walkforward_baseline_v1::wf_...,soxlbacktest_qlib_walkforward_baseline_v1,qlib_converted_buysell_reference,wf_pass_ytd18_15_15_02,3,1,False,1,False,False,...,83.509497,trend_capture,wf_2020,trend_capture,-30.618705,12.785207,60.618705,0.000000,60.618705,83.509497
5,soxlbacktest_qlib_walkforward_baseline_v1::low...,soxlbacktest_qlib_walkforward_baseline_v1,qlib_converted_buysell_reference,low_risk_15_05_02,3,1,False,0,False,False,...,88.256963,trend_capture,wf_2020,trend_capture,-32.131801,6.828971,62.131801,0.000000,62.131801,88.256963
6,release_gate_bc_official_replay_v1::baseline_c...,release_gate_bc_official_replay_v1,release_gate_bc_weighting,baseline_current,3,0,False,0,False,False,...,89.265484,joint_tradeoff,wf_2020,joint_tradeoff,15.901974,24.834087,14.098026,4.834087,62.438895,89.265484
7,release16_state_head_post_fullprobe_objective_...,release16_state_head_post_fullprobe_objective_...,state_head_objective_screen,full_actor_probe,3,0,False,1,False,False,...,99.196903,joint_tradeoff,wf_2020,trend_capture,-22.718827,19.260466,52.718827,0.000000,52.718827,99.196903
8,release_gate_bc_official_replay_v1::release64_...,release_gate_bc_official_replay_v1,release_gate_bc_weighting,release64_no_objective,3,0,False,0,False,False,...,103.083505,joint_tradeoff,wf_2022,drawdown_control,62.750639,26.860565,0.000000,6.860565,68.605646,103.083505
9,soxlbacktest_qlib_walkforward_baseline_v1::low...,soxlbacktest_qlib_walkforward_baseline_v1,qlib_converted_buysell_reference,low_risk_20_08_02,3,0,False,0,False,False,...,126.786339,joint_tradeoff,wf_2020,trend_capture,-24.165599,9.073791,54.165599,0.000000,54.165599,126.786339


## Conclusion

The final cell renders `summary.json`. This handoff is not a promotion; it is the next objective definition for a future training/checkpoint screen.


In [6]:

summary = load_json(CARD / "summary.json")
lines = [
    f"Decision: `{summary['decision']}`.",
    f"Best candidate: `{summary['pool_summary']['best_candidate_id']}`; failure profile: `{summary['pool_summary']['best_failure_profile']}`.",
    f"Full-gate candidates at <=20% MaxDD: {summary['pool_summary']['full_gate_count_20dd']} / {summary['pool_summary']['candidate_count']}; strict <=15%: {summary['pool_summary']['full_gate_count_15dd']}.",
    f"Handoff decision: `{summary['training_objective_handoff']['decision']}` with terms `{summary['training_objective_handoff']['objective_terms']}`.",
    f"Next mechanism: `{summary['next_mechanism']}`.",
    summary["conclusion"],
]
display(Markdown("\n".join(f"- {line}" for line in lines)))


- Decision: `official_candidate_pool_no_full_gate_training_objective_handoff`.
- Best candidate: `release16_drawdown_upside_checkpoint_replay_v1::bc_initialization`; failure profile: `trend_capture`.
- Full-gate candidates at <=20% MaxDD: 0 / 16; strict <=15%: 0.
- Handoff decision: `train_rebound_participation_objective` with terms `['walkforward_relative_return_gap_penalty', 'walkforward_drawdown_gap_penalty', 'high_vol_rebound_participation_floor']`.
- Next mechanism: `train_rebound_participation_objective`.
- No trainable non-reference candidate passes the official walk-forward pool gate; next step is the emitted training/checkpoint objective handoff, not PPO length or actor retention.